# 第9回　ワークブック

企業の資金調達・財務三表とJ-Quantsの決算取得

2026年11月5日（木）2限　金融論

## 使い方

上から順に、1つずつ実行していくnotebookです。

- 実行する　code cellを選び、Shift+Enterで実行します。押す前に、何が出るか予想します。
- やってみる　すぐ下の `# ここに書きます` のcellに自分で書いて実行します。既にあるcellを書き換える指示もあります。

1〜5節は、小さな架空の数値で運転資金と財務三表を計算します。API keyは使いません。6節で、J-Quantsから1社の決算サマリーを取得し、決算短信と照合します。6節の最後の確認メモは、HW02の段階提出Aの「決算短信の数値確認メモ」に使えます。

詰まったら、cellの内容とエラーの表示をCodexに見せて聞けます。「やってみる」の答え合わせも頼めます。

`workbook.py` には、「実行する」のcodeが上から順に入っていて、各節に会計の意味を書いたコメントが付いています。terminalで `python workbook.py` と打つと単独で実行でき、6節でAPI keyの入力欄が出ます。

## 準備

1. `workbook/` folderを、自分の作業場所にコピーします。
2. VS Codeでfolderごと開き、このnotebookで kernel `finance-env` を選びます。
3. 下のcellを実行します。pathに `finance-env` が含まれ、pandasのversionが表示されれば準備完了です。
4. 6節には、第8回で発行したJ-QuantsのAPI keyが要ります。実行するたびにAPIから取り直します。取得したデータは、そのままの形で他の人に配布しません。

In [1]:
import sys
print(sys.executable)

import pandas as pd
print(pd.__version__)

[検証に使用したPython]
2.2.3


# 1. 運転資金と設備資金

## 1-1　仕入れ・販売・入金の時差

会計は「売ったとき・使ったときに記録する」（発生主義）で、「現金が動いたときに記録する」（現金主義）ではありません。そのため、利益と現金は同じ月には動きません。この節では、そのずれを数値で見ます。

架空の小売店です。単位は万円です。

- 1〜4か月目に、毎月80万円の商品を仕入れ、同じ月に100万円で売ります。在庫は残りません。
- 仕入れの代金は、1か月後に払います（買掛金）。
- 売上の代金は、2か月後に受け取ります（売掛金）。
- 5か月目からは仕入れも販売もせず、残りの代金のやり取りだけをします。

実行する

In [2]:
cash_flow = pd.DataFrame({"month": [1, 2, 3, 4, 5, 6]})
cash_flow["sales"] = [100, 100, 100, 100, 0, 0]
cash_flow["purchases"] = [80, 80, 80, 80, 0, 0]
cash_flow["profit"] = cash_flow["sales"] - cash_flow["purchases"]
cash_flow["cash_in"] = cash_flow["sales"].shift(2, fill_value=0)
cash_flow["cash_out"] = cash_flow["purchases"].shift(1, fill_value=0)
cash_flow["cash_change"] = cash_flow["cash_in"] - cash_flow["cash_out"]
cash_flow["profit_total"] = cash_flow["profit"].cumsum()
cash_flow["cash_total"] = cash_flow["cash_change"].cumsum()
cash_flow

,month,sales,purchases,profit,cash_in,cash_out,cash_change,profit_total,cash_total
0,1,100,80,20,0,0,0,20,0
1,2,100,80,20,0,80,-80,40,-80
2,3,100,80,20,100,80,20,60,-60
3,4,100,80,20,100,80,20,80,-40
4,5,0,0,0,100,80,20,80,-20
5,6,0,0,0,100,0,100,80,80


`profit` は「売上 − 仕入れ」で、売った月に発生主義で記録します。`cash_in`・`cash_out` は、実際に現金が動く月です。`shift(2, fill_value=0)` は2行下へずらし、空いた行を0にします。売上が2か月後に入金になる、という時差を表しています。`cumsum()` は上からの累計です。

利益は毎月20万円出ています。一方、現金の累計 `cash_total` は、2か月目の終わりに-80万円です。仕入れの支払いが先に来て、売上の入金が後になるからです。この不足を借入などで埋められなければ、利益が出ていても支払いができません。これが黒字倒産です。6か月目に代金のやり取りが終わると、現金の累計は利益の累計と同じ80万円に追いつきます。

やってみる

売上の入金を1か月後に早めます。`shift(2, fill_value=0)` を `shift(1, fill_value=0)` に変えて実行すると、`cash_total` は0、20、40、60、80、80となり、現金が足りない月がなくなります。確かめたら `shift(2, fill_value=0)` に戻して、もう一度実行します。

In [3]:
# ここに書きます

## 1-2　運転資金＝売掛金＋在庫－買掛金

売掛金は、売ったがまだ受け取っていない代金です。相手に貸しているのと同じで、資産です。買掛金は、仕入れたがまだ払っていない代金です。相手から借りているのと同じで、負債です。在庫は、売るために持っている商品で、これも資産です。

運転資金は、商品と売掛金の形で寝ている資金から、取引先に待ってもらっている買掛金を引いたものです。営業を回すために手元に置いておく必要がある資金で、この金額の分だけ、利益が現金になっていません。

実行する

In [4]:
cash_flow["receivables"] = cash_flow["sales"].cumsum() - cash_flow["cash_in"].cumsum()
cash_flow["inventory"] = 0
cash_flow["payables"] = cash_flow["purchases"].cumsum() - cash_flow["cash_out"].cumsum()
cash_flow["working_capital"] = cash_flow["receivables"] + cash_flow["inventory"] - cash_flow["payables"]
cash_flow["profit_minus_wc"] = cash_flow["profit_total"] - cash_flow["working_capital"]
cash_flow[["month", "profit_total", "receivables", "payables", "working_capital", "profit_minus_wc", "cash_total"]]

,month,profit_total,receivables,payables,working_capital,profit_minus_wc,cash_total
0,1,20,100,80,20,0,0
1,2,40,200,80,120,-80,-80
2,3,60,200,80,120,-60,-60
3,4,80,200,80,120,-40,-40
4,5,80,100,0,100,-20,-20
5,6,80,0,0,0,80,80


`receivables` は「売上の累計 − 入金の累計」で、その月末に未回収で残っている売掛金です。`payables` は「仕入れの累計 − 支払いの累計」で、未払いで残っている買掛金です。2か月目の終わりは、売掛金200万円、買掛金80万円で、運転資金は120万円です。

`profit_minus_wc`（利益の累計－運転資金）は、どの月も `cash_total` と同じです。利益が出ていても、その分が売掛金として相手の手元にある間は、現金になっていません。

やってみる

売上と利益はそのままで、販売に備えて商品を多めに持ち、在庫が毎月10万円ずつ増えて40万円で止まる場合を考えます。1-1と1-2のcodeを下のcellに写して、売上原価 `cogs` と仕入れ `purchases` を分けます。仕入れは売上原価に在庫の増加分を足した額で、翌月に払います。4か月目は買掛金90万円、運転資金150万円、累積現金−70万円です。在庫を持つと、その分の資金が商品の形でとどまり、必要な資金が増えます。

In [5]:
# ここに書きます

# 2. 内部資金と外部資金

ある会社が、200百万円の機械を買う資金を集めます。単位は百万円です。

集めた資金は、B/Sの右側に記録されます。利益の留保と株式の発行は純資産に、借入は負債に入ります。純資産には返済の義務がなく、負債には期限に返す義務があります。これが、同じ200百万円でも資金の性格が違うところです。

実行する

In [6]:
funding = pd.DataFrame({
    "source": ["retained earnings", "new shares", "bank loan"],
    "kind": ["internal", "external", "external"],
    "bs_side": ["net assets", "net assets", "liabilities"],
    "amount": [60, 40, 100],
})
funding

,source,kind,bs_side,amount
0,retained earnings,internal,net assets,60
1,new shares,external,net assets,40
2,bank loan,external,liabilities,100


内部資金は、会社が稼いだ利益のうち、配当などで外に出さずに残した分（利益の留保）です。外部資金は、株式の発行、社債、銀行からの借入など、会社の外から集めた資金です。

実行する

In [7]:
total = funding["amount"].sum()
internal = funding[funding["kind"] == "internal"]["amount"].sum()
liabilities_increase = funding[funding["bs_side"] == "liabilities"]["amount"].sum()
net_assets_increase = funding[funding["bs_side"] == "net assets"]["amount"].sum()

print("total:", total, " internal share:", round(internal / total * 100, 1), "%")
print("machine acquired by exchanging 200 cash; acquisition itself does not increase net assets")
print("liabilities + net assets increase:", liabilities_increase + net_assets_increase)

total: 200  internal share: 30.0 %
machine acquired by exchanging 200 cash; acquisition itself does not increase net assets
liabilities + net assets increase: 200


内部資金の割合は30.0%です。B/Sでは、資産が機械の200百万円増え、右側は負債が100百万円、純資産が100百万円増えます。左右の増加は同じ200百万円です。

利益の留保60百万円で純資産は増えましたが、その60百万円は機械の代金に使われ、現金としては残っていません。B/Sの利益剰余金は、過去の利益のうち会社に残した額の記録であって、現金の残高ではありません。

やってみる

`amount` を `[60, 80, 60]` に変えます（新株を80、借入を60）。内部資金の割合は30.0%のままで、負債の増加は60、純資産の増加は140、合計は200です。

In [8]:
# ここに書きます

# 3. B/S：ある時点の資産・負債・純資産

## 3-1　資産＝負債＋純資産

期末のB/Sです。単位は百万円です。

B/S（貸借対照表）は、ある時点の残高です。左側の資産は資金の使い道、右側の負債と純資産は資金をどこから集めたかを表します。資産はすべて負債か純資産のどちらかで賄われているので、左右の合計は必ず一致します。

実行する

In [9]:
balance = pd.DataFrame({
    "item": ["cash", "receivables", "inventory", "machines",
             "payables", "bank loans",
             "capital", "retained earnings", "non-controlling interests"],
    "side": ["assets", "assets", "assets", "assets",
             "liabilities", "liabilities",
             "net assets", "net assets", "net assets"],
    "amount": [50, 120, 30, 200, 80, 150, 100, 60, 10],
})

assets = balance[balance["side"] == "assets"]["amount"].sum()
liabilities = balance[balance["side"] == "liabilities"]["amount"].sum()
net_assets = balance[balance["side"] == "net assets"]["amount"].sum()
print("assets:", assets)
print("liabilities + net assets:", liabilities + net_assets)
balance

assets: 400
liabilities + net assets: 400


,item,side,amount
0,cash,assets,50
1,receivables,assets,120
2,inventory,assets,30
3,machines,assets,200
4,payables,liabilities,80
5,bank loans,liabilities,150
6,capital,net assets,100
7,retained earnings,net assets,60
8,non-controlling interests,net assets,10


資産400百万円＝負債230百万円＋純資産170百万円です。1年以内に現金になる現金・売掛金・在庫が流動資産、それより長く使う機械が固定資産です。1年以内に払う買掛金が流動負債です。

純資産には、子会社の株式のうち親会社以外の株主の分（非支配株主持分）10百万円が入っています。純資産からこれを除いた160百万円が、親会社の株主の分（自己資本）です。

実行する

In [10]:
equity = net_assets - 10
print("net assets / assets:", round(net_assets / assets * 100, 1), "%")
print("equity / assets:", round(equity / assets * 100, 1), "%")

net assets / assets: 42.5 %
equity / assets: 40.0 %


自己資本比率（自己資本÷総資産）は、借入に頼らずにどれだけ資産を賄っているかの目安で、高いほど返済の負担が小さい会社です。純資産を総資産で割ると42.5%、自己資本を総資産で割ると40.0%です。6節で見るJ-Quantsの `Eq` は純資産、`EqAR` は自己資本比率なので、`Eq / TA` と `EqAR` は一致しないことがあります。

やってみる

銀行から50百万円を借りて、現金のまま持ちます。`cash` を100、`bank loans` を200に変えると、資産450＝負債280＋純資産170になり、自己資本比率は35.6%に下がります。

In [11]:
# ここに書きます

## 3-2　簿価の純資産と時価総額

この会社の発行済株式数を10万株、株価を2,500円とします。

実行する

In [12]:
shares = 100_000
price = 2500
market_cap = price * shares / 1_000_000
book_equity = 160
print("market capitalization (million yen):", market_cap)
print("book equity (million yen):", book_equity)
print("book equity per share (yen):", book_equity * 1_000_000 / shares)

market capitalization (million yen): 250.0
book equity (million yen): 160
book equity per share (yen): 1600.0


時価総額は250百万円、B/Sの自己資本は160百万円です。B/Sの金額は、過去に払い込まれた資本と積み上げた利益を帳簿の価格で記録したもの（簿価）です。時価総額は、市場で付いた株価に株数を掛けたもの（時価）です。B/Sは過去の取引の記録、時価総額は投資家の将来の予想を映すので、2つが一致する理由はありません。1株当たりの簿価1,600円は、6節の `BPS` にあたります。この比を使う指標（PBR）は、第10回で扱います。

やってみる

株価を1,200円に変えます。時価総額は120百万円になり、簿価の自己資本160百万円より小さくなります。B/Sの数値は変わりません。

In [13]:
# ここに書きます

# 4. P/L：期間の収益と費用

1年間の損益です。単位は百万円です。

P/L（損益計算書）は、期間中に収益をいくら得て、費用をいくら使ったかの流れです。B/Sが時点の残高であるのに対し、P/Lは期間の増減です。日本基準のP/Lは、利益を上から「本業に近い順」に段階ごとに表示します。

実行する

In [14]:
sales = 1000
cost_of_sales = 700
sga = 220
non_operating_income = 15
non_operating_expenses = 25
extraordinary_losses = 10
income_taxes = 18
profit_to_nci = 2

gross_profit = sales - cost_of_sales
operating_profit = gross_profit - sga
ordinary_profit = operating_profit + non_operating_income - non_operating_expenses
profit_before_tax = ordinary_profit - extraordinary_losses
net_income = profit_before_tax - income_taxes
net_income_parent = net_income - profit_to_nci

steps = pd.DataFrame({
    "line": ["売上高", "売上総利益", "営業利益", "経常利益",
             "税金等調整前当期純利益", "当期純利益", "親会社株主に帰属する当期純利益"],
    "million_yen": [sales, gross_profit, operating_profit, ordinary_profit,
                    profit_before_tax, net_income, net_income_parent],
})
steps

,line,million_yen
0,売上高,1000
1,売上総利益,300
2,営業利益,80
3,経常利益,70
4,税金等調整前当期純利益,60
5,当期純利益,42
6,親会社株主に帰属する当期純利益,40


売上高から売上原価を引くと売上総利益300、販売費及び一般管理費（`sga`）を引くと営業利益80です。営業利益は本業の利益です。受取利息などの営業外収益を足し、支払利息などの営業外費用を引くと、経常利益70になります。災害損失のような臨時の損益（特別損益）と税金を引いた当期純利益42のうち、親会社の株主の分が40です。

段階を分けるのは、「本業で稼げているか」（営業利益）と「利息や一時的な損で最終利益がどう変わったか」を分けて読むためです。営業利益は黒字なのに純利益が赤字なら、本業以外のどこかに原因があります。

経常利益は日本基準の決算にある段階です。IFRS（国際会計基準）の損益計算書には経常利益の行がなく、6節のトヨタのように、J-Quantsの `OdP` は空欄になります。

やってみる

販管費 `sga` を220から240に変えます。営業利益は60、経常利益は50、親会社株主に帰属する当期純利益は20です（税金は18のままとします）。

In [15]:
# ここに書きます

# 5. C/Fと減価償却

## 5-1　200,000円の機械を4年間使う

C/F（キャッシュ・フロー計算書）は、期間中の現金の出入りを、営業活動・投資活動・財務活動の3つに分けます。P/Lは発生主義、C/Fは現金主義なので、両者のずれを見ると利益と現金の違いが分かります。

減価償却は、何年も使う設備の代金を、買った年に全部費用にするのではなく、使う年数に分けて費用にする記録の仕方です。設備が売上を生む期間と、その費用を対応させるためです。

- 会社は、株主が払い込んだ200,000円の現金（資本金）で始まります。
- 1年目の初めに、200,000円の機械を現金で買います。耐用年数は4年、4年後の価値（残存価額）は0です。
- 機械を使って、毎年80,000円を現金で売り上げます。ほかの費用と税金はないものとします。
- 減価償却は定額法で、毎年 200,000 ÷ 4 = 50,000円を費用にします。

実行する

In [16]:
price = 200000
life = 4
years = list(range(1, life + 1))    # [1, 2, 3, 4]

pl = pd.DataFrame({"year": years})
pl["sales"] = 80000
pl["depreciation"] = price / life
pl["net_income"] = pl["sales"] - pl["depreciation"]
pl

,year,sales,depreciation,net_income
0,1,80000,50000.0,30000.0
1,2,80000,50000.0,30000.0
2,3,80000,50000.0,30000.0
3,4,80000,50000.0,30000.0


P/Lでは、毎年の費用は減価償却費50,000円だけです。純利益は毎年30,000円です。

実行する

In [17]:
cf = pd.DataFrame({"year": years})
cf["operating"] = pl["net_income"] + pl["depreciation"]
cf["investing"] = [-price] + [0] * (life - 1)    # 1年目だけ -200,000
cf["financing"] = 0
cf["change_in_cash"] = cf["operating"] + cf["investing"] + cf["financing"]
cf

,year,operating,investing,financing,change_in_cash
0,1,80000.0,-200000,0,-120000.0
1,2,80000.0,0,0,80000.0
2,3,80000.0,0,0,80000.0
3,4,80000.0,0,0,80000.0


営業活動によるキャッシュ・フロー `operating` は、純利益に減価償却費を足し戻した80,000円です。減価償却費はP/Lでは費用ですが、その年に現金は1円も出ていきません（出たのは1年目の購入時です）。だから純利益に足し戻すと、本業で実際に入った現金になります。実際の決算書の営業C/Fも、純利益から出発して減価償却費などを足し戻す形（間接法）で作られています。機械の代金200,000円は、1年目の投資活動によるキャッシュ・フロー `investing` に出ます。借入や配当はないので、財務活動 `financing` は0です。

実行する

In [18]:
bs = pd.DataFrame({"year": years})
bs["cash"] = price + cf["change_in_cash"].cumsum()
bs["accumulated_depreciation"] = pl["depreciation"].cumsum()
bs["machine_book_value"] = price - bs["accumulated_depreciation"]
bs["total_assets"] = bs["cash"] + bs["machine_book_value"]
bs["liabilities"] = 0
bs["capital"] = price
bs["retained_earnings"] = pl["net_income"].cumsum()
bs["liabilities_and_net_assets"] = bs["liabilities"] + bs["capital"] + bs["retained_earnings"]
bs["balanced"] = bs["total_assets"] == bs["liabilities_and_net_assets"]
bs

,year,cash,accumulated_depreciation,machine_book_value,total_assets,liabilities,capital,retained_earnings,liabilities_and_net_assets,balanced
0,1,80000.0,50000.0,150000.0,230000.0,0,200000,30000.0,230000.0,True
1,2,160000.0,100000.0,100000.0,260000.0,0,200000,60000.0,260000.0,True
2,3,240000.0,150000.0,50000.0,290000.0,0,200000,90000.0,290000.0,True
3,4,320000.0,200000.0,0.0,320000.0,0,200000,120000.0,320000.0,True


各年末のB/Sです。機械の帳簿の価格（簿価）は、減価償却の累計（減価償却累計額）だけ毎年減り、4年目の終わりに0になります。現金は80,000円ずつ増えます。純資産は、資本金200,000円に純利益の累計（利益剰余金）を足したものです。`balanced` はすべて `True` で、どの年も資産＝負債＋純資産です。

同じ機械の代金200,000円が、3つの表に別々の形で出ています。C/Fでは買った年の投資活動に現金の支出として一度に、P/Lでは4年間に分けて減価償却費として、B/Sでは機械の簿価が毎年50,000円ずつ減る形で。純資産が毎年増えるのは純利益が利益剰余金に加わるからで、減価償却費は費用であって純資産に足すものではありません。

やってみる

耐用年数を5年にします。5-1の3つのcellの内容を下のcellにまとめて写し、`life = 5` に変えて実行します。表の名前を `pl5`、`cf5`、`bs5` にすると、基準の4年の表は上書きされません。減価償却費は40,000円、純利益は40,000円になり、5年目の機械の簿価は0、`balanced` はすべて `True` です。

In [19]:
# ここに書きます

## 5-2　機械の代金を買った年に全部費用にすると

実行する

In [20]:
compare = pd.DataFrame({"year": years})
compare["net_income_with_depreciation"] = pl["net_income"]
compare["change_in_cash"] = cf["change_in_cash"]
compare["all_expensed_in_year1"] = [80000 - price] + [80000] * (life - 1)
print(compare)
print(compare.sum())

   year  net_income_with_depreciation  change_in_cash  all_expensed_in_year1
0     1                       30000.0       -120000.0                -120000
1     2                       30000.0         80000.0                  80000
2     3                       30000.0         80000.0                  80000
3     4                       30000.0         80000.0                  80000
year                                10.0
net_income_with_depreciation    120000.0
change_in_cash                  120000.0
all_expensed_in_year1           120000.0
dtype: float64


減価償却をすると、純利益は毎年30,000円です。機械の代金を1年目に全部費用にすると、1年目は-120,000円、2〜4年目は80,000円になり、現金の増減と同じ形になります。4年間の合計は、3つとも120,000円で同じです。違うのは、どの年に記録するかです。減価償却は、4年間使う機械の費用を4年に分け、その年の売上と対応させる記録の仕方であって、現金の動きを変えるものではありません。

やってみる

1年目の売上だけが80,000円から0円に減った場合（機械の立ち上げに1年かかった場合）を考えます。純利益は1年目が-50,000円、2〜4年目が30,000円です。現金の増減は、1年目が-200,000円です。

In [21]:
# ここに書きます

# 6. 決算サマリーの取得と照合

## 6-1　1社の決算サマリーを取得する

ここからは実在の会社の決算の数値を扱い、1〜5節で見た項目が実際の決算ではどの列に入るかを確かめます。取得のcodeは第8回の `toyota_oct2025.py` と同じ仕組みで、下のcellに書いてあります。`fetch` がURLに条件を付けてJ-Quantsに送り、返ってきたJSONをDataFrameにします。API keyは環境変数 `JQUANTS_API_KEY` にあればそれを使い、なければ最初の取得のときに非表示の入力欄で受け取ります。実行するたびにAPIへ問い合わせるので、開示が追加・訂正されていれば、行や数値が前回と変わります。

`/fins/summary` に銘柄コードを送ると、決算短信の1ページ目（サマリー）の数値が「開示1回＝1行」で返ります。

実行する

In [ ]:
# 第8回の `toyota_oct2025.py`と同じ仕組みです。URLに条件を付けて送ると、JSONが返ってきます。
# その "data" の中身（1行が1つの辞書）をDataFrameにします。
# API keyは、環境変数 JQUANTS_API_KEY にあればそれを使い、なければ最初の取得のときに
# 非表示の入力欄で受け取ります。このfileにも出力にも書き込みません。
# 実行するたびにAPIから取り直します。開示が追加・訂正されていれば、行や数値が前回と変わります。
# 取得したデータは、J-Quantsの利用条件により、そのままの形で他の人に配布しません。
#
# 応答が200以外のとき fetch は「HTTP 4xx」で止まります。よくある原因は次のとおりです。
#   400  銘柄コード・日付・取得できる期間（Freeプランは12週間前まで）が違う
#   401  API keyが違う
#   403  このプランでは取れないデータか期間
#   429  回数制限。Freeプランは1分に5回まで。少し待ってからやり直す
#
# 行を選んで Shift+Enter で使うときは、先に下の import と BASE・api_key の行を送り、
# 次に def の行から関数の終わりまでをまとめて送ります。関数の中の数行だけを送ると SyntaxError になります。
# 1行が長かったり、複数の関数を一度に選んだりすると、terminalの行編集が途中で崩れることがあります。
# 一度送った定義はそのterminalを閉じるまで残るので、以後は fetch(...) の行だけを送れば済みます。
import os
from getpass import getpass

import requests

BASE = "https://api.jquants.com/v2"
api_key = os.environ.get("JQUANTS_API_KEY", "")


def fetch(endpoint, params):
    """J-Quants API V2に条件を送り、返ってきた全行をDataFrameにする。"""
    global api_key
    if not api_key:
        api_key = getpass("J-Quants API key: ").strip()
    params = dict(params)
    rows = []
    while True:
        r = requests.get(BASE + endpoint, params=params,
                         headers={"x-api-key": api_key}, timeout=30)
        if r.status_code != 200:
            raise RuntimeError(f"HTTP {r.status_code}")
        payload = r.json()
        rows.extend(payload["data"])
        cursor = payload.get("pagination_key")
        if not cursor:
            break
        params["pagination_key"] = cursor
    if not rows:
        raise RuntimeError("0件です。銘柄・期間を確認します。")
    return pd.DataFrame(rows)


In [ ]:
code = "72030"   # 証券コード7203の末尾に0を付けた5桁
fins = fetch("/fins/summary", {"code": code})

# 読んだ直後は、日付も金額も文字列なので、使う列を日付型と数値型に直す（第7回と同じ手順）
for col in ["DiscDate", "CurPerSt", "CurPerEn", "CurFYSt", "CurFYEn"]:
    fins[col] = pd.to_datetime(fins[col], errors="coerce")
for col in ["Sales", "OP", "OdP", "NP", "TA", "Eq", "EqAR", "EPS", "BPS", "DivAnn"]:
    fins[col] = pd.to_numeric(fins[col], errors="coerce")
fins = fins.sort_values(["DiscDate", "CurPerEn"]).reset_index(drop=True)

print(fins.shape)
fins[["DiscDate", "DocType", "CurPerType", "CurPerSt", "CurPerEn"]]

トヨタ自動車（証券コード7203、APIでは `72030`）の決算短信のサマリーが、開示日 `DiscDate` の順に並びます。1行が1回の開示です。2026年9月17日に取得したときは、2024年8月1日から2026年5月8日までの8行でした。Freeプランで取得できるのは12週間より前の約2年分なので、取得する日によって最初と最後の行が変わります。

`DocType` は書類の種類です。`FYFinancialStatements_Consolidated_IFRS` は、通期（FY）・連結（Consolidated）・IFRSの決算短信です。親会社だけの決算（単体）なら `NonConsolidated`、日本基準なら末尾が `JP` になります。`CurPerType` は期間の種類（1Q、2Q、3Q、FY）、`CurPerSt` と `CurPerEn` は期間の初日と末日です。

やってみる

`fins["DocType"].str.contains("_Consolidated_")` を実行します。8行すべてが `True` で、トヨタの行はどれも連結の決算です。

In [34]:
# ここに書きます
fins["DocType"].str.contains("_Consolidated_")

0    True
1    True
2    True
3    True
4    True
5    True
6    True
7    True
Name: DocType, dtype: bool

## 6-2　四半期は期首からの累計

実行する

In [25]:
fy2026 = fins[fins["CurFYEn"] == "2026-03-31"]
fy2026[["DiscDate", "CurPerType", "CurPerSt", "CurPerEn", "Sales", "OP", "NP"]]

,DiscDate,CurPerType,CurPerSt,CurPerEn,Sales,OP,NP
4,2025-08-07,1Q,2025-04-01,2025-06-30,12253326000000,1166141000000,841345000000
5,2025-11-05,2Q,2025-04-01,2025-09-30,24630753000000,2005692000000,1773426000000
6,2026-02-06,3Q,2025-04-01,2025-12-31,38087604000000,3196722000000,3030891000000
7,2026-05-08,FY,2025-04-01,2026-03-31,50684952000000,3766216000000,3848098000000


決算短信の四半期の数値は、その四半期だけでなく、事業年度の初めからの累計です。2026年3月期（2025年4月1日〜2026年3月31日）の4行です。`CurPerSt` はどの行も2025-04-01で、3Qの `Sales` は4月から12月までの9か月分の累計です。金額の単位は円です。`Sales` は売上高の列で、トヨタの決算短信では「営業収益」にあたります。`OP` は営業利益、`NP` は4節の最後の段階の親会社株主に帰属する当期純利益で、当期純利益そのものではありません。

やってみる

1〜3月の3か月だけの売上を、FYの `Sales` から3Qの `Sales` を引いて計算します。12,597,348,000,000円（約12.6兆円）です。

In [35]:
# ここに書きます
fy_sales = fy2026[fy2026["CurPerType"] == "FY"]["Sales"].iloc[0]
q3_sales = fy2026[fy2026["CurPerType"] == "3Q"]["Sales"].iloc[0]
print(fy_sales - q3_sales)


12597348000000


## 6-3　実績と予想

実行する

In [27]:
for col in ["FSales", "FNP", "NxFSales", "NxFNp"]:
    print(col, fins[col].dtype)
    fins[col] = pd.to_numeric(fins[col], errors="coerce")

view = fins[["DiscDate", "CurPerType"]].copy()
for col in ["Sales", "FSales", "NxFSales", "NP", "FNP", "NxFNp"]:
    view[col + " (100M yen)"] = (fins[col] / 100_000_000).round(0)
view

FSales object
FNP object
NxFSales object
NxFNp object


,DiscDate,CurPerType,Sales (100M yen),FSales (100M yen),NxFSales (100M yen),NP (100M yen),FNP (100M yen),NxFNp (100M yen)
0,2024-08-01,1Q,118379.0,460000.0,NaN,13333.0,35700.0,NaN
1,2024-11-06,2Q,232824.0,460000.0,NaN,19071.0,35700.0,NaN
2,2025-02-05,3Q,356735.0,470000.0,NaN,41004.0,45200.0,NaN
3,2025-05-08,FY,480367.0,NaN,485000.0,47651.0,NaN,31000.0
4,2025-08-07,1Q,122533.0,485000.0,NaN,8413.0,26600.0,NaN
5,2025-11-05,2Q,246308.0,490000.0,NaN,17734.0,29300.0,NaN
6,2026-02-06,3Q,380876.0,500000.0,NaN,30309.0,35700.0,NaN
7,2026-05-08,FY,506850.0,NaN,510000.0,38481.0,NaN,30000.0


6-1で数に直したのは実績の列だけなので、予想の列 `FSales`・`FNP`・`NxFSales`・`NxFNp` はまだ `object`（文字列）です。同じように `pd.to_numeric(..., errors="coerce")` で数にし、空欄は `NaN` にしています。表示は億円です。

決算短信には、実績と会社の業績予想の両方が載っています。`Sales` や `NP` は実績です。四半期の行の `FSales`・`FNP` は、その事業年度の通期について会社が出した予想です。予想は会社が出す見通しで、株価はこの予想の変化にも反応します（第10回）。通期（FY）の行では `FSales` は空欄で、翌事業年度の予想が `NxFSales`・`NxFNp` に入ります。2026年5月8日の行では、2026年3月期の売上高の実績が506,850億円、2027年3月期の予想が510,000億円です。

やってみる

2025年8月7日（1Q）から2026年2月6日（3Q）までの3行で、通期の純利益の予想 `FNP` がどう変わったかを見ます。26,600億円、29,300億円、35,700億円と上がり、実績の `NP` は38,481億円でした。

In [28]:
# ここに書きます

## 6-4　業績予想の修正と、同じ期間の2つの行

トヨタには、決算短信以外の行がありません。他の会社では、決算短信とは別の日に業績予想だけを修正した行（`DocType` が `EarnForecastRevision`）や、同じ期間の行が2つあることがあります。

やってみる

1. `nissan = fetch("/fins/summary", {"code": "72010"})` で日産自動車を取得します。2026年9月17日の取得では、17行のうち9行が `EarnForecastRevision` です。これらの行は実績の `Sales` が空欄で、予想の列（通期の `FSales`、第2四半期までの `FSales2Q` など）だけが入っています。2026年3月期の `NP` は-533,095,000,000円の赤字です。
2. `aeon = fetch("/fins/summary", {"code": "82670"})` でイオンを取得します。2025年11月30日までの3Qの行が、開示日2026-01-08と2026-01-14の2つあり、数値は同じです。開示番号 `DiscNo` で区別できます。2つ目の行が出た理由は、このサマリーだけでは分かりません。企業のIRページの開示一覧で確かめます。

In [29]:
# ここに書きます

## 6-5　最新の通期の行

実行する

In [30]:
fy = fins[(fins["CurPerType"] == "FY")
          & fins["DocType"].str.startswith("FYFinancialStatements_Consolidated_")]
if fy.empty:
    raise ValueError("連結の通期決算の行がありません")
latest_fy = fy.sort_values(["CurPerEn", "DiscDate"]).iloc[-1]
latest_fy[["DiscDate", "DocType", "CurPerEn", "Sales", "OP", "OdP", "NP", "TA", "Eq", "EqAR", "EPS", "BPS", "DivAnn"]]

DiscDate                        2026-05-08 00:00:00
DocType     FYFinancialStatements_Consolidated_IFRS
CurPerEn                        2026-03-31 00:00:00
Sales                                50684952000000
OP                                    3766216000000
OdP                                             NaN
NP                                    3848098000000
TA                                  105522331000000
Eq                                   41020068000000
EqAR                                          0.378
EPS                                          295.25
BPS                                         3062.82
DivAnn                                         95.0
Name: 7, dtype: object

通期（`FY`）・連結の決算短信の行を、決算期末の順に並べた最後の行です。IFRSのトヨタには経常利益の行がないので、`OdP` は `NaN` です。

`TA` は総資産（B/Sの左側の合計）、`Eq` は純資産、`EqAR` は自己資本比率です。`Eq / TA` は約0.389で、`EqAR` の0.378と一致しません。3-1と同じく、純資産には親会社以外の株主の分が入っているからです。`EPS` は1株当たり純利益（`NP`÷株数）、`BPS` は1株当たり純資産（自己資本÷株数。3-2の1株当たりの簿価）、`DivAnn` は1株当たりの年間配当です。

やってみる

`latest_fy["Eq"] / latest_fy["TA"]` と `latest_fy["EqAR"] * latest_fy["TA"] / 1_000_000` を計算します。2つ目は約39,887,441百万円で、決算短信の「親会社の所有者に帰属する持分」39,918,854百万円に近い値です（`EqAR` が小数第3位までに丸められているので、ぴったりは一致しません）。

In [31]:
# ここに書きます

## 6-6　決算短信と照合する

トヨタの決算短信は、[トヨタ自動車の決算情報のページ](https://global.toyota/jp/ir/financial-results/)から開けます。2026年3月期の決算短信（2026年5月8日）は、過去の決算資料の一覧にあります。1ページ目の「(1)連結経営成績」と「(2)連結財政状態」に、百万円単位で数値が載っています。

APIの値が正しいかは、原資料である決算短信と突き合わせて確かめます。確かめることは、数値が一致するかと、APIの列名が決算短信のどの項目にあたるかの2つです。下の `tanshin` の `None` に、決算短信の数値を百万円のまま入れて実行すると、`difference` にAPIとの差が出ます。

実行する

In [32]:
tanshin = {
    "Sales": None,   # 営業収益
    "OP": None,      # 営業利益
    "NP": None,      # 親会社の所有者に帰属する当期利益
    "TA": None,      # 資産合計
    "Eq": None,      # 資本合計
}

rows = []
for item in ["Sales", "OP", "NP", "TA", "Eq"]:
    rows.append({
        "item": item,
        "api (million yen)": latest_fy[item] / 1_000_000,
        "tanshin (million yen)": tanshin[item],
    })
check = pd.DataFrame(rows)
check["difference"] = check["api (million yen)"] - pd.to_numeric(check["tanshin (million yen)"])
check

,item,api (million yen),tanshin (million yen),difference
0,Sales,50684952.0,None,NaN
1,OP,3766216.0,None,NaN
2,NP,3848098.0,None,NaN
3,TA,105522331.0,None,NaN
4,Eq,41020068.0,None,NaN


`None` のままでは `tanshin` の列が空欄で、`difference` は `NaN` です。

やってみる

決算短信の1ページ目から、営業収益、営業利益、親会社の所有者に帰属する当期利益、資産合計、資本合計を読み、上のcellの `None` を数値に置き換えて実行します。トヨタの2026年3月期なら、5項目とも `difference` は0.0です。`NP` は「当期利益」（3,985,761百万円）ではなく「親会社の所有者に帰属する当期利益」と、`Eq` は「親会社の所有者に帰属する持分」（39,918,854百万円）ではなく「資本合計」と一致します。

In [33]:
# ここに書きます

## 6-7　確認メモ

照合の結果を、次のcellに書きます。このメモは、HW02の段階提出Aの「決算短信の数値確認メモ」に使えます。自分のteamの企業で同じことをするときは、6-1の `code` を変えて6節を上から実行し直します。

## 決算短信の数値確認メモ

- 企業名・銘柄コード：
- 対象の決算期（`CurPerEn`）と開示日（`DiscDate`）：
- 書類の種類（`DocType`）：通期／四半期、連結／単体、会計基準
- 照合した原資料（URL、ページ、表の名前）：
- 照合した項目と結果（APIの値、決算短信の値、単位、一致したか）：
- APIのサマリーの列と、決算短信の項目名の対応で気づいたこと：
- サマリーにない数値で、原資料で読んだこと：

# 7. 最初から実行し直す

1. notebook上部の Restart を押します。
2. Run All を押します。6-1でもう一度APIから取得します。kernelを再起動したので、API keyの入力欄も出ます。
3. 6-4のやってみるで日産とイオンを取得した場合、APIの呼び出しは毎回3回になります。Freeプランの回数制限は1分に5回なので、続けて実行し直すときは少し間を空けます。

1〜5節の表は、同じ数値でもう一度できます。6節の表は、同じ期間の開示が追加・訂正されていなければ同じです。